In [1]:
# =======================================
# CELL 1: Imports and Configuration
# =======================================
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../results/supplementary')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORIES = ['general_help_seeking', 'evidence_based_treatments', 
              'specific_conditions']

COUNTRIES = ['UA',
             'DE', 'FR', 'PL', 'NL', 'LV', 'LT', 'EE', 
             'US', 'GB', 'AU', 'CA',
             'GE', 'AM', 'AZ', 'KZ', 'UZ', 'MD', 'IL', 'JP', 'BR']

EXPECTED_MONTHS = 96  # Jan 2018 - Dec 2025

# Find the latest raw data folder (matching other notebooks)
raw_folders = sorted(DATA_DIR.glob('raw_*'))
if not raw_folders:
    raise FileNotFoundError(f'No raw_* folders found in {DATA_DIR}')
LATEST = raw_folders[-1]
print(f'Analyzing data folder: {LATEST.name}')
print(f'Countries: {len(COUNTRIES)}, Categories: {len(CATEGORIES)}')


# =======================================
# CELL 2: Analysis helper
# =======================================
def analyze_country_category(country_dir: Path, category: str) -> dict:
    """Analyze data quality for one (country, category) combination."""
    csv_files = sorted(country_dir.glob(f'{category}__*__chunk*.csv'))
    
    result = {
        'n_chunks': len(csv_files),
        'n_keywords_total': 0,
        'n_keywords_with_data': 0,
        'months_observed': set(),
        'n_partial_observations': 0,
        'languages_seen': set(),
        'no_data_keywords': []
    }
    
    if not csv_files:
        return result
    
    for f in csv_files:
        # Parse filename: {category}__{language_or_id}__chunk{N}.csv
        parts = f.stem.split('__')
        language = parts[1] if len(parts) >= 3 else 'unknown'
        result['languages_seen'].add(language)
        
        try:
            df = pd.read_csv(f, parse_dates=['date'], index_col='date')
            
            if 'isPartial' in df.columns:
                result['n_partial_observations'] += int(df['isPartial'].sum())
                df = df[df['isPartial'] == False].drop(columns=['isPartial'])
            
            df = df.replace(0, np.nan)
            
            result['n_keywords_total'] += df.shape[1]
            
            all_nan_cols = df.columns[df.isna().all()].tolist()
            for kw in all_nan_cols:
                result['no_data_keywords'].append((kw, language))
            
            result['n_keywords_with_data'] += df.shape[1] - len(all_nan_cols)
            df = df.drop(columns=all_nan_cols)
            
            if df.shape[1] > 0:
                non_nan_rows = df.dropna(how='all').index
                for date in non_nan_rows:
                    result['months_observed'].add(date.to_period('M'))
                    
        except Exception as e:
            print(f'  Warning: {f.name}: {e}')
    
    result['n_months_observed'] = len(result['months_observed'])
    result['n_months_missing'] = EXPECTED_MONTHS - result['n_months_observed']
    result['pct_complete'] = round(
        (result['n_months_observed'] / EXPECTED_MONTHS) * 100, 1
    )
    return result


# =======================================
# CELL 3: Run analysis across all (country × category)
# =======================================
all_results = []
no_data_global = []

for country in COUNTRIES:
    country_dir = LATEST / country
    if not country_dir.exists():
        print(f'WARNING: {country} folder not found')
        continue
    
    for category in CATEGORIES:
        analysis = analyze_country_category(country_dir, category)
        
        all_results.append({
            'country': country,
            'category': category,
            'n_chunks': analysis['n_chunks'],
            'n_keywords_total': analysis['n_keywords_total'],
            'n_keywords_with_data': analysis['n_keywords_with_data'],
            'n_months_observed': analysis['n_months_observed'],
            'n_months_missing': analysis['n_months_missing'],
            'pct_complete': analysis['pct_complete'],
            'n_partial_observations': analysis['n_partial_observations'],
            'languages': ', '.join(sorted(analysis['languages_seen']))
        })
        
        for kw, lang in analysis['no_data_keywords']:
            no_data_global.append({
                'country': country,
                'category': category,
                'keyword': kw,
                'language': lang
            })

df_summary = pd.DataFrame(all_results)
df_no_data = pd.DataFrame(no_data_global)

print(f'\nProcessed {len(df_summary)} country-category combinations.')
print(f'Mean completeness: {df_summary["pct_complete"].mean():.1f}%')
print(f'Keywords with no data: {len(df_no_data)}')


# =======================================
# CELL 4: Save outputs
# =======================================
df_summary.to_csv(OUTPUT_DIR / 'table_s1_completeness.csv', index=False)
df_no_data.to_csv(OUTPUT_DIR / 'table_s2_no_data_keywords.csv', index=False)

summary_stats = {
    'data_collection_folder': LATEST.name,
    'expected_months': EXPECTED_MONTHS,
    'n_countries_analyzed': len(COUNTRIES),
    'n_categories': len(CATEGORIES),
    'overall': {
        'mean_pct_complete': round(df_summary['pct_complete'].mean(), 1),
        'min_pct_complete': float(df_summary['pct_complete'].min()),
        'max_pct_complete': float(df_summary['pct_complete'].max()),
        'total_keywords_attempted': int(df_summary['n_keywords_total'].sum()),
        'total_keywords_with_data': int(df_summary['n_keywords_with_data'].sum()),
        'total_partial_observations': int(df_summary['n_partial_observations'].sum()),
        'total_no_data_keywords': len(df_no_data)
    },
    'no_data_by_category': df_no_data.groupby('category').size().to_dict() if len(df_no_data) > 0 else {}
}

with open(OUTPUT_DIR / 'supplementary_summary.json', 'w') as f:
    json.dump(summary_stats, f, indent=2, default=str)

print(f'\nSaved to {OUTPUT_DIR}/')
print('  - table_s1_completeness.csv')
print('  - table_s2_no_data_keywords.csv')
print('  - supplementary_summary.json')

Analyzing data folder: raw_20260417_211800
Countries: 21, Categories: 3

Processed 63 country-category combinations.
Mean completeness: 77.8%
Keywords with no data: 411

Saved to ..\results\supplementary/
  - table_s1_completeness.csv
  - table_s2_no_data_keywords.csv
  - supplementary_summary.json
